# Revision experiments

The analyses and runs added in response to the reviewers' seven points. The notebook works on the experiment folder on Google Drive (`runs/elner/`, `review_pooled.csv` and the Wikidata cache). It never deletes anything and backs up the review file before changing it.

| Point | Section | Model cost |
|---|---|---|
| 2. Is the NIL check just a fixed rule? | A | none |
| 6. Results by script | B | none |
| 5a. Rejected NIL labels and item creation dates | C | none (Wikidata only) |
| 7. Coverage of the data file | D | none (Hugging Face download) |
| 4. Baselines on the same retrieval without an LLM | E | none |
| 1. A general, ungrounded critic in the same loop | F | about 1,900 calls |
| 3. Repeated runs | G | about 1,000 to 1,100 calls per run |
| 3. An open-weights model | H | free on a Colab GPU, but slow |
| 5b. Initial answers the reviewer had not seen | I | manual review |
| Final tables and report | J | none |
| Extra numbers (dates, every repair and break) | K | none |

Sections A to E are free. F and G need Gemini API calls, H needs a GPU session and I is a manual review. Section J writes `revision_report.txt` and section K `revision_report_extra.txt`, both in `runs/elner/revision/`.

In [ ]:
!pip install -q google-genai anthropic openai ijson huggingface_hub datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. The module
The notebook uses `elagent.py` from this repository. The revision added:
- `verify_critic` and the configuration `critic_only`: a general LLM critic in the style of DeepEL's self-validation. It reviews **every** answer, NIL included, from the text alone, inside the same correction loop.
- `grounded_only`: the Wikidata checks without the LLM coherence judgment.
- `baseline_exact_popularity`: the same retrieval without an LLM; an exact name match first, then the item with the most sitelinks.
- `apply_nil_rule`, `nil_rule_analysis`, `by_script`, `unseen_initial`, `nil_label_timing`, `corpus_profile` and `multi_run_summary`.
- `export_pooled_review(..., include_initial=True)`: the reviewer also judges the initial answers.

Earlier runs are unaffected: the earlier configurations behave exactly as in version 1.0.0.

In [ ]:
# Use elagent.py from this repository (cloned once into the Colab session).
import os, sys
REPO_URL = "https://github.com/MahammediAnesAbdelouahab/arabic-el-agent.git"
REPO_DIR = "/content/arabic-el-agent"
if not os.path.exists(os.path.join(REPO_DIR, "elagent.py")):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

In [ ]:
import os, sys, io, contextlib, importlib, json, shutil, datetime
PROJECT_DIR = "/content/drive/MyDrive/arabic_el_agent"
sys.path.insert(0, REPO_DIR)
import elagent
importlib.reload(elagent)
from elagent import *
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print("elagent version:", ELAGENT_VERSION)
assert ELAGENT_VERSION >= "1.1.0", "old module loaded: Runtime > Restart session, then run again"

## Setup
Put your e-mail address in `user_agent`, as the Wikimedia API policy asks. The API key is read from Colab Secrets.

In [ ]:
CFG = Config(
    project_dir=PROJECT_DIR,
    llm_backend="gemini",
    llm_model="gemini-3.8-flash",        # the model used for every run in the paper
    min_llm_interval=0.0,
    user_agent="ArabicELAgent/0.1 (academic research; contact: your_email@example.com)",   # <- your e-mail
)
from google.colab import userdata
API_KEY = userdata.get("GEMINI_API_KEY")
wd = WikidataClient(CFG.user_agent, cache_path=os.path.join(PROJECT_DIR, "cache", "wikidata_cache.json"))
llm = LLM(CFG.llm_backend, CFG.llm_model, api_key=API_KEY, temperature=CFG.temperature,
          thinking_level=CFG.gemini_thinking_level, min_interval=CFG.min_llm_interval)

EXP_DIR = os.path.join(PROJECT_DIR, "runs", "elner")
POOLED_CSV = os.path.join(EXP_DIR, "review_pooled.csv")
AGREE_CSV = os.path.join(EXP_DIR, "review_agreement_B.csv")
OUT_DIR = os.path.join(EXP_DIR, "revision")
os.makedirs(OUT_DIR, exist_ok=True)

all_rows = load_rows(os.path.join(PROJECT_DIR, "data", "elner_sample.jsonl"))
dev_rows, test_rows = split_rows(all_rows, dev_frac=0.2, seed=13)
print("dev:", len(dev_rows), "| test:", len(test_rows))      # expected: 211 | 850

KNOWN = set(ABLATIONS) | {"llm_plus_nil_rule"}


def list_runs(complete_only=False):
    """{run name: path} for every run file of the test set (dev runs and the stale memory run are left out)."""
    out = {}
    for f in sorted(os.listdir(EXP_DIR)):
        if not f.endswith(".jsonl") or f.startswith("dev_"):
            continue
        name = f[:-6]
        if name.split("@")[0] not in KNOWN or name.startswith("full_with_memory"):
            continue
        if complete_only:
            n = sum(1 for r in load_rows(os.path.join(EXP_DIR, f)) if r.get("qid") != "ERROR")
            if n < len(test_rows):
                print(f"not complete, left out: {name} ({n}/{len(test_rows)})")
                continue
        out[name] = os.path.join(EXP_DIR, f)
    return out


def first_runs(names):
    runs = list_runs()
    return {n: runs[n] for n in names if n in runs}


ACC, UNCLEAR, SUMM = pooled_gold(POOLED_CSV)
print("reviewed mentions:", len(ACC), "| unclear:", len(UNCLEAR))
print("runs found:", ", ".join(list_runs()))
REPORT = []          # every result below is also appended here, then written to revision_report.txt


def report(title, obj):
    text = obj.to_string() if hasattr(obj, "to_string") else json.dumps(obj, ensure_ascii=False, indent=1, default=str)
    REPORT.append(f"=== {title} ===\n{text}\n")
    print(f"=== {title} ===")
    display(obj) if hasattr(obj, "to_string") else print(text)

## A. The NIL check against a fixed rule (point 2)
Why a correction loop rather than a rule that says: if the model answers NIL and a candidate matches the mention exactly with a compatible type, choose that candidate?

- **The first cell** reads the agent's traces. On the mentions where the check fired, what did the agent do, and what would the rule have done?
- **The second cell** builds a complete system, `llm_plus_nil_rule`, from the LLM-only run without any new model call, and compares it with the other systems.

In [ ]:
agent_recs = load_rows(os.path.join(EXP_DIR, "full_no_memory.jsonl"))
df_orig, s_orig = nil_rule_analysis(agent_recs)
df_corr, s_corr = nil_rule_analysis(agent_recs, acceptable=ACC, exclude=UNCLEAR)
report("NIL check vs fixed rule (original labels)", s_orig)
report("NIL check vs fixed rule (corrected labels)", s_corr)
report("NIL check cases (corrected labels)", df_corr)
df_corr.to_csv(os.path.join(OUT_DIR, "nil_rule_cases.csv"), index=False)

In [ ]:
rule_run = apply_nil_rule(load_rows(os.path.join(EXP_DIR, "llm_no_correction.jsonl")), wd)
save_rows(rule_run, os.path.join(EXP_DIR, "llm_plus_nil_rule.jsonl"))
sys_names = ["baseline_search_top1", "llm_no_correction", "llm_plus_nil_rule", "full_no_memory"]
report("LLM + fixed NIL rule vs agent (original labels)",
       paired_table(first_runs(sys_names), ref="llm_no_correction"))
report("LLM + fixed NIL rule vs agent (corrected labels; the rule's new answers are judged in section I)",
       paired_table(first_runs(sys_names), ref="llm_no_correction", acceptable=ACC, exclude=UNCLEAR))

## B. Results by script (point 6)
A mention counts as Latin-script if it contains Latin letters. This is the same criterion that makes the agent also search in English and French.

In [ ]:
main_names = ["baseline_search_top1", "llm_no_correction", "full_no_memory", "full_plus_role"]
report("By script (original labels)", by_script(first_runs(main_names), ref="llm_no_correction"))
report("By script (corrected labels)", by_script(first_runs(main_names), ref="llm_no_correction",
                                                 acceptable=ACC, exclude=UNCLEAR))
report("Mentions per script in the test set",
       pd.Series([mention_script(r["mention"]) for r in test_rows]).value_counts())

## C. Were the rejected NIL labels wrong when the data were made? (point 5a)
For every NIL label that the review rejected, the cell fetches the creation date of the accepted Wikidata item. ELNER-DZ version 2 was published on 25 June 2025. An item created after that date did not exist when the label was made, so NIL was right at the time.

One Wikidata request per item; the answers are cached.

In [ ]:
timing, counts = nil_label_timing(POOLED_CSV, wd, cutoff="2025-06-25")
report("Wrong NIL labels: was the accepted item created before the dataset's publication (2025-06-25)?", counts)
report("Wrong NIL labels with creation dates", timing)
timing.to_csv(os.path.join(OUT_DIR, "nil_label_timing.csv"), index=False)

## D. Does the sample represent the corpus? (point 7)
The sample was drawn from a local copy of the Zenodo data file that ended early. This section:
1. counts the records read from the local copy and compares them with the 2,075,148 rows of the Hugging Face release;
2. streams the full corpus from Hugging Face, without loading it into memory;
3. compares three profiles (full corpus, part that was read, our sample): share of NIL labels, share of Latin script, entity types and sentence length;
4. checks that every sampled sentence appears in the Hugging Face release with the same text.

Reading the full corpus takes a few minutes.

In [ ]:
ELNER_DIR = os.path.join(PROJECT_DIR, "data", "elner_dz")
local_files = sorted(os.path.join(d, f) for d, _, fs in os.walk(ELNER_DIR) for f in fs
                     if f.lower().endswith((".json", ".jsonl", ".ndjson", ".csv", ".tsv", ".conll", ".txt", ".gz")))
print("local files:", local_files)
schema = detect_schema(list(iter_records(local_files[0], limit=300)))
print("schema:", schema)
prof_local = corpus_profile((r for f in local_files for r in iter_records(f)), schema)
print("records read from your copy:", prof_local["sentences"])

In [ ]:
HF_REPO = "HadjerHaninebgt7878/ELNER-DZ"
try:
    from datasets import load_dataset
    hf = load_dataset(HF_REPO, split="train", streaming=True)
    first_hf = next(iter(hf))
except Exception as e:                       # fallback: download the raw files and read them like the Zenodo copy
    print("datasets could not stream the repository:", repr(e)[:200], "-> downloading the files instead")
    from huggingface_hub import list_repo_files, hf_hub_download
    files = [f for f in list_repo_files(HF_REPO, repo_type="dataset") if f.lower().endswith((".json", ".jsonl", ".gz"))]
    print("data files:", files)
    paths = [hf_hub_download(HF_REPO, f, repo_type="dataset") for f in files]

    class _Stream:                            # re-iterable, like a streaming dataset
        def __iter__(self):
            return (r for p in paths for r in iter_records(p))
    hf = _Stream()
    first_hf = next(iter(hf))
print("first HF record:", json.dumps(first_hf, ensure_ascii=False)[:400])
schema_hf = detect_schema([r for _, r in zip(range(300), iter(hf))])
prof_full = corpus_profile(iter(hf), schema_hf)
prof_sample = corpus_profile(all_rows)
coverage = prof_local["sentences"] / prof_full["sentences"]
report("Corpus profiles (full = Hugging Face; read = your Zenodo copy; sample = our 1,000 sentences)",
       pd.DataFrame({"full corpus": prof_full, "read part": prof_local, "our sample": prof_sample}))
report("Coverage of your copy", {"records read": prof_local["sentences"], "records in full corpus": prof_full["sentences"],
                                 "coverage": round(coverage, 4)})

In [ ]:
# Are our sampled sentences in the Hugging Face version, with the same text?
want = {str(r["doc_id"]): r["text"] for r in all_rows}
found = same = 0
for rec in hf:
    rid = str(rec.get(schema_hf.get("id") or "id"))
    if rid in want:
        found += 1
        same += (rec.get(schema_hf["text"]) == want[rid])
        if found == len(want):
            break
report("Our sample in the Hugging Face version", {"sampled sentences": len(want), "found by id": found,
                                                   "identical text": same})

## E. Baselines on the same retrieval without an LLM (point 4)
- `baseline_popularity`: the agent's candidate with the most sitelinks.
- `baseline_exact_popularity`: an exact name match first, then the most sitelinks.

No model calls; the Wikidata cache is used. The only difference from the LLM system's retrieval is query expansion, which needs the model.

In [ ]:
base = run_ablations(test_rows, CFG, wd, llm, names=["baseline_popularity", "baseline_exact_popularity"], out_dir=EXP_DIR)
report("Retrieval-matched baselines (raw)", base)

## F. A general, ungrounded critic in the same loop (point 1)
- `critic_only`: the same loop (three rounds, two searches, confirmation allowed), but the only checks are the hard format checks and a general LLM critic that reviews every answer from the text alone.
- `grounded_only`: the Wikidata checks alone, without the coherence judgment.

Approximate cost: `critic_only` needs about 2.2 calls per mention (about 1,900 calls) and `grounded_only` about 1.3 (about 1,100). Runs resume after an interruption.

In [ ]:
new = run_ablations(test_rows, CFG, wd, llm, names=["critic_only", "grounded_only"], out_dir=EXP_DIR)
report("New configurations (raw, original labels)", new)

## G. Repeated runs (point 3)
The first run already exists; this section adds runs tagged `@r2`. The Wikidata cache makes retrieval identical, so differences between runs come from the model alone.

**Every agent run is also an LLM-only run:** the initial decision is produced exactly as `llm_no_correction` produces its answer (same candidates, same prompt, same searches); the checks come afterwards. Section J therefore measures LLM-only variation from the first answers of every complete run.

Approximate cost per run: 1,000 calls for `llm_no_correction` and 1,100 for `full_no_memory`. Incomplete runs are left out of every table.

In [ ]:
PLAN = [("llm_no_correction", "@r2"), ("full_no_memory", "@r2")]    # + ("full_no_memory", "@r3") if the budget allows
for name, tag in PLAN:
    t = run_ablations(test_rows, CFG, wd, llm, names=[name], out_dir=EXP_DIR, tag=tag)
    report(f"Repeated run {name}{tag} (raw, original labels)", t)

## H. An open-weights model (point 3)
**Run this section in a separate GPU session** (Runtime > Change runtime type > T4 GPU), after the setup cells.

It uses Ollama inside Colab, with no key and no cost. One run of `full_no_memory` is enough: its first answer is the LLM-only system and its final answer is the agent.

1. **Server:** restarts Ollama with an 8,192-token context, so that long prompts are not cut silently, and keeps the model on the GPU.
2. **Probe (about a minute):** is the model on the GPU, is it still "thinking", and how long will the full run take?
3. **Run:** resumes where it stopped if the session ends (rerun cells 1 and 3 only).

**Model choice:** the paper used `qwen3:4b-instruct-2507-q4_K_M` (2.5 GB, no thinking mode). `qwen3:8b` also fits on a T4 but is slower. Do not use the bare `qwen3:4b` tag, which points to the thinking variant.

With a hosted provider of open models that offers an OpenAI-compatible API, set `BASE_URL`, `OPEN_MODEL` and `OPEN_API_KEY`; the server cell then skips Ollama.

In [ ]:
RUN_OPEN_MODEL = False                       # True in a GPU session
OPEN_MODEL = "qwen3:4b-instruct-2507-q4_K_M"   # the model used in the paper; "qwen3:8b" also fits a T4
BASE_URL = "http://localhost:11434/v1"       # Ollama's OpenAI-compatible endpoint
OPEN_API_KEY = "ollama"                      # a hosted provider: userdata.get("...")

import subprocess, time, shutil, urllib.request


def sh(cmd, check=True):
    """Run a shell command and show its output (the error message too, if it fails)."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout or "")[-1500:], (r.stderr or "")[-1500:])
    if check and r.returncode:
        raise RuntimeError(f"failed with code {r.returncode}: {cmd}")


def ollama_works(path):
    try:
        return bool(path) and subprocess.run([path, "--version"], capture_output=True).returncode == 0
    except OSError:                                # e.g. a broken binary left by an interrupted install
        return False


if RUN_OPEN_MODEL and BASE_URL.startswith("http://localhost:11434"):
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("No GPU in this session: Runtime > Change runtime type > T4 GPU, then rerun the start cells")
    sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv")
    sh("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw", check=False)   # the installer needs zstd
    OLLAMA = shutil.which("ollama")
    if not ollama_works(OLLAMA):
        if OLLAMA and os.path.exists(OLLAMA):
            os.remove(OLLAMA)
        sh("curl -fsSL https://ollama.com/install.sh | sh")
        OLLAMA = shutil.which("ollama") or "/usr/local/bin/ollama"
    subprocess.run(["pkill", "-f", "ollama serve"])                 # restart with the settings below
    time.sleep(2)
    env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192",             # long prompts are not cut off
               OLLAMA_KEEP_ALIVE="24h")                              # the model stays on the GPU between calls
    subprocess.Popen([OLLAMA, "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):                      # wait until the server answers
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError("Ollama did not start; see /content/ollama.log")
    sh(f"{OLLAMA} pull {OPEN_MODEL}")

In [ ]:
if RUN_OPEN_MODEL:
    llm_open = LLM("openai", OPEN_MODEL, api_key=OPEN_API_KEY, base_url=BASE_URL, system_suffix=" /no_think")
    print(llm_open.json(SYSTEM_PROMPT, 'Reply with JSON: {"ok": true}'))      # warm-up: loads the model

    if BASE_URL.startswith("http://localhost:11434"):
        ps = subprocess.run([shutil.which("ollama"), "ps"], capture_output=True, text=True).stdout
        print(ps)
        if "100% GPU" not in ps:
            print("⚠️  the model is not fully on the GPU -> it will be very slow")

    r = llm_open._client.chat.completions.create(          # is the model still "thinking"?
        model=OPEN_MODEL, messages=[{"role": "system", "content": SYSTEM_PROMPT + " /no_think"},
                                    {"role": "user", "content": 'Reply with JSON: {"ok": true}'}])
    toks = r.usage.completion_tokens if r.usage else None
    print("tokens for a one-line JSON answer:", toks, "⚠️  thinking is on" if toks and toks > 150 else "(ok)")

    N = 8
    probe = Agent(replace(CFG, **ABLATIONS["full_no_memory"]), wd, llm_open, None)
    c0, s0, t0 = llm_open.calls, llm_open.seconds, time.time()
    for row in test_rows[:N]:
        probe.link_document([Mention(text=row["text"], mention=row["mention"], type=row.get("type"),
                                     start=row.get("start"), end=row.get("end"), id=row.get("id"))])
    calls = llm_open.calls - c0
    per_mention = (time.time() - t0) / N
    print(f"{(llm_open.seconds - s0) / max(1, calls):.1f} s per call | {calls / N:.2f} calls per mention | "
          f"{per_mention:.1f} s per mention")
    print(f"estimate for full_no_memory on {len(test_rows)} mentions: {per_mention * len(test_rows) / 3600:.1f} h")
    if os.path.exists("/content/ollama.log") and "truncat" in open("/content/ollama.log").read().lower():
        print("⚠️  Ollama cut a prompt short: raise OLLAMA_CONTEXT_LENGTH in the cell above")

In [ ]:
if RUN_OPEN_MODEL:
    OPEN_CONFIGS = ["full_no_memory"]      # one run = two systems: first answer = LLM only, final answer = agent
    # OPEN_CONFIGS += ["critic_only"]      # only if time allows (about 1.7x longer)
    tag = "@" + OPEN_MODEL.replace(":", "-").replace("/", "-")
    t = run_ablations(test_rows, CFG, wd, llm_open, names=OPEN_CONFIGS, out_dir=EXP_DIR, tag=tag)
    report(f"Open-weights model {OPEN_MODEL} (raw, original labels)", t)

## I. Judging the new answers and the initial answers (point 5b)
**First, before any review:** how many repairs or breaks rest on an initial answer that the reviewer never saw?

**Then the review:** every answer not judged yet is shown to the reviewer, blind: the answers of the new systems, of the repeated runs, and the initial answers. Earlier verdicts are kept as long as the options shown have not changed.

- `REJUDGE_CARRIED = True` reopens the verdicts carried over from the earlier non-blind pass, so that the whole review is blind.
- The current review file is first backed up as `review_pooled_before_revision_<date>.csv`.
- **Only complete runs are included:** a partial run would drop rows from the file.

In [ ]:
before = unseen_initial(first_runs(["full_no_memory", "full_plus_role"]), POOLED_CSV)
report("Repairs/breaks resting on initial answers the reviewer never saw (before re-review)", before)

In [ ]:
REJUDGE_CARRIED = True

stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M")
backup = os.path.join(EXP_DIR, f"review_pooled_before_revision_{stamp}.csv")
shutil.copy(POOLED_CSV, backup)
print("backup:", backup)
if REJUDGE_CARRIED:
    rv = pd.read_csv(POOLED_CSV, dtype=str, keep_default_na=False)
    mask = rv["carried"] == "yes"
    rv.loc[mask, ["status", "accepted", "annotator", "carried"]] = ""
    rv.to_csv(POOLED_CSV, index=False, encoding="utf-8-sig")
    print("reopened carried verdicts:", int(mask.sum()))

review_runs = list_runs(complete_only=True)
print("runs in the review:", ", ".join(review_runs))
export_pooled_review(review_runs, POOLED_CSV, wd, audit=40, include_initial=True)
review_blind(POOLED_CSV, annotator="A")

## J. Final tables and report
Run after the review. The cell computes every table with the corrected labels and writes `revision_report.txt` to `runs/elner/revision/`.

In [ ]:
ACC, UNCLEAR, SUMM = pooled_gold(POOLED_CSV)
report("Label review summary (after revision)", pd.DataFrame(SUMM).T)
if os.path.exists(AGREE_CSV):
    report("Inter-annotator agreement (A after revision vs B)", agreement(POOLED_CSV, AGREE_CSV))

runs = list_runs(complete_only=True)
first = {k: v for k, v in runs.items() if "@" not in k}
for labels, kw in (("original labels", {}), ("corrected labels", dict(acceptable=ACC, exclude=UNCLEAR))):
    report(f"All systems, first run ({labels})", paired_table(first, ref="llm_no_correction", **kw))

for tag in sorted({"@" + k.split("@", 1)[1] for k in runs if "@" in k and not k.endswith(("@r2", "@r3"))}):
    sub = {k: v for k, v in runs.items() if k.endswith(tag)}
    report(f"Open-weights model {tag} (corrected labels)",
           paired_table(sub, ref="llm_no_correction" + tag, acceptable=ACC, exclude=UNCLEAR))

per, agg = multi_run_summary(runs, ["llm_no_correction", "full_no_memory", "critic_only", "grounded_only"],
                             ["", "@r2", "@r3"], acceptable=ACC, exclude=UNCLEAR)
report("Repeated runs: one row per run (corrected labels)", per)
report("Repeated runs: summary (corrected labels)", agg)

# Each run's FIRST answer is an independent LLM-only run (same procedure), so the run-to-run spread of the
# LLM-only system is measured on every complete run of the main model, not only on llm_no_correction.
LLM_ONLY_FROM = ("llm_no_correction", "full_no_memory", "full_plus_role", "grounded_only", "critic_only")
main = {k: v for k, v in runs.items() if k.split("@")[0] in LLM_ONLY_FROM and ("@" not in k or k.endswith(("@r2", "@r3")))}
with contextlib.redirect_stdout(io.StringIO()):
    pt = paired_table(main, acceptable=ACC, exclude=UNCLEAR, n_boot=0)
report("LLM-only accuracy = first answer of every run (corrected labels)",
       pt[["n", "initial_accuracy", "accuracy", "fixed", "broken", "sign_test_p"]])
report("LLM-only run-to-run spread over these runs (corrected labels)",
       pt["initial_accuracy"].astype(float).describe().round(4))

for name in ["full_no_memory", "grounded_only", "critic_only", "full_plus_role"]:
    if name in runs:
        report(f"What each check did in {name} (corrected labels)",
               fix_attribution(load_rows(runs[name]), acceptable=ACC, exclude=UNCLEAR))

_, s_corr = nil_rule_analysis(load_rows(runs["full_no_memory"]), acceptable=ACC, exclude=UNCLEAR)
report("NIL check vs fixed rule (corrected labels, after revision)", s_corr)
report("Repairs/breaks resting on unseen initial answers (after re-review)",
       unseen_initial({k: v for k, v in first.items() if k.startswith(("full", "critic", "grounded"))}, POOLED_CSV))
report("By script (corrected labels)", by_script(first, ref="llm_no_correction", acceptable=ACC, exclude=UNCLEAR))
report("By popularity (corrected labels)", paired_by(first, "category", ref="llm_no_correction",
                                                     acceptable=ACC, exclude=UNCLEAR))
timing, counts = nil_label_timing(POOLED_CSV, wd, cutoff="2025-06-25")
report("Wrong NIL labels vs item creation date (after revision)", counts)

path = os.path.join(OUT_DIR, "revision_report.txt")
open(path, "w", encoding="utf-8").write(f"elagent {ELAGENT_VERSION} | {datetime.datetime.now():%Y-%m-%d %H:%M}\n\n"
                                        + "\n".join(REPORT))
wd.cache.save()
print("report written:", path)

## K. Extra numbers
No model calls; a few seconds. **Run section D first in the same session** (it streams the corpus from Hugging Face), then this cell. It writes `revision_report_extra.txt` to `runs/elner/revision/`.

- What the rejected labels are: by type, and how many are years (`2015`, `2022`, ...).
- The main results again **without DATE mentions and years**, since ELNER-DZ leaves dates unlinked.
- Every repair and break in the agent, rule and Qwen runs, and the kinds of breaks of the general critic.

In [ ]:
import re, io, contextlib
EXTRA = []


def rep(title, obj):
    text = obj.to_string() if hasattr(obj, "to_string") else json.dumps(obj, ensure_ascii=False, indent=1, default=str)
    EXTRA.append(f"=== {title} ===\n{text}\n")
    print(f"=== {title} ===\n{text}\n")


ACC, UNCLEAR, SUMM = pooled_gold(POOLED_CSV)
runs = list_runs(complete_only=True)
row_of = {str(r["id"]): r for r in test_rows}
gold = {i: r["gold"] for i, r in row_of.items()}
t = pd.DataFrame({"id": list(row_of)})
t["type"] = [str(row_of[i].get("type") or "-") for i in t["id"]]
t["year"] = [bool(re.fullmatch(r"\s*[0-9\u0660-\u0669]{4}\s*", row_of[i]["mention"])) for i in t["id"]]
t["gold_nil"] = [gold[i] == "NIL" for i in t["id"]]
t["reviewed"] = [i in ACC for i in t["id"]]
t["unclear"] = [i in UNCLEAR for i in t["id"]]
t["gold_wrong"] = [i in ACC and gold[i] not in ACC[i] for i in t["id"]]
by_type = t.groupby("type")[["year", "gold_nil", "reviewed", "unclear", "gold_wrong"]].sum()
by_type.insert(0, "mentions", t.groupby("type").size())
rep("Test mentions by type", by_type)
rep("Wrong labels by original label (gold_nil), type and year",
    t[t.gold_wrong].groupby(["gold_nil", "type", "year"]).size().rename("n").reset_index())

DATES = set(t.loc[t["type"].str.upper().isin(["DATE", "TIME"]) | t["year"], "id"])
print("DATE/year mentions:", len(DATES))
MAIN = ["baseline_search_top1", "llm_no_correction", "llm_plus_nil_rule", "full_no_memory", "full_no_memory@r2",
        "grounded_only", "critic_only", "full_plus_role"]
main = {k: runs[k] for k in MAIN if k in runs}
qwen = {k: v for k, v in runs.items() if "qwen" in k.lower()}
for title, ex in (("all mentions", set(UNCLEAR)), ("without DATE/year mentions", set(UNCLEAR) | DATES)):
    with contextlib.redirect_stdout(io.StringIO()):
        rep(f"Corrected labels, {title}", paired_table(main, ref="llm_no_correction", acceptable=ACC, exclude=ex))
        if qwen:
            rep(f"Qwen, corrected labels, {title}", paired_table(qwen, acceptable=ACC, exclude=ex))
rep("What each check did in full_no_memory, without DATE/year mentions",
    fix_attribution(load_rows(runs["full_no_memory"]), acceptable=ACC, exclude=set(UNCLEAR) | DATES))


def changes(name):
    out = []
    for r in load_rows(runs[name]):
        i = str(r["id"])
        if i in UNCLEAR or r["qid"] == "ERROR":
            continue
        acc = ACC.get(i, {gold[i]})
        a, b = r.get("initial_qid") in acc, r["qid"] in acc
        if a != b:
            out.append({"run": name, "change": "repaired" if b else "broke", "id": i, "mention": r["mention"],
                        "type": r.get("type"), "initial": r.get("initial_qid"), "final": r["qid"],
                        "accepted": "|".join(sorted(acc))})
    return out


names = [n for n in ["full_no_memory", "full_no_memory@r2", "grounded_only", "llm_plus_nil_rule"] if n in runs] + list(qwen)
rep("Every repair and break (corrected labels)", pd.DataFrame([c for n in names for c in changes(n)]))
cc = pd.DataFrame(changes("critic_only")) if "critic_only" in runs else pd.DataFrame()
if len(cc):
    cc["kind"] = ["NIL -> QID" if a == "NIL" else "QID -> NIL" if b == "NIL" else "QID -> other QID"
                  for a, b in zip(cc["initial"], cc["final"])]
    rep("critic_only: repairs and breaks by kind (corrected labels)", cc.groupby(["change", "kind"]).size())

from_d = [x for x in REPORT if x.startswith(("=== Corpus profiles", "=== Coverage", "=== Our sample"))]
if not from_d:
    print("⚠️  section D (corpus profile) was not run in this session: run it, then this cell again")
path = os.path.join(OUT_DIR, "revision_report_extra.txt")
open(path, "w", encoding="utf-8").write(f"elagent {ELAGENT_VERSION} | {datetime.datetime.now():%Y-%m-%d %H:%M}\n\n"
                                        + "\n".join(from_d + EXTRA))
print("written:", path)